# Projeto 1 — Análise Exploratória (EDA) e Métricas para Dashboard

Base: ReclameAqui • **Ibyte**  
Fonte do dado tratado: `data/prc/reclameaqui_ibyte_clean.csv`

Este notebook cobre os pontos do **Acompanhamento 2** (estatística e EDA) e já deixa prontas as principais **métricas e agregações** para o dashboard (série temporal, Pareto por UF, mapa, proporções de status e análises de texto).


## Objetivos de análise (sugestão)

Perguntas norteadoras para a consultoria analítica:

1. **Volume e tendência:** como evolui o número de reclamações ao longo do tempo? Há sazonalidade?
2. **Geografia:** quais estados concentram mais reclamações? (Pareto e mapa)
3. **Causas:** quais categorias/temas aparecem com mais frequência?
4. **Efetividade:** qual a distribuição de `status` e como ela varia por categoria e por UF?
5. **Texto (NLP básico):** o tamanho do texto muda conforme o `status`? Quais termos são mais frequentes nas descrições?


In [ ]:
from __future__ import annotations

from collections import Counter
from pathlib import Path
import json
import re
import unicodedata
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from wordcloud import WordCloud

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)
sns.set_theme(style="whitegrid")


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for parent in [start, *start.parents]:
        if (parent / "data").exists() and (parent / "docs").exists():
            return parent
    return start


In [ ]:
PROJECT_ROOT = find_repo_root(Path.cwd())
RAW_PATH = PROJECT_ROOT / "data" / "raw" / "RECLAMEAQUI_IBYTE.csv"
PRC_PATH = PROJECT_ROOT / "data" / "prc" / "reclameaqui_ibyte_clean.csv"

RAW_PATH, PRC_PATH

In [ ]:
if PRC_PATH.exists():
    df = pd.read_csv(PRC_PATH, encoding="utf-8")
else:
    raise FileNotFoundError(
        "Arquivo processado não encontrado. Rode antes o notebook 01 (limpeza): "
        f"{PRC_PATH}"
    )

# Tipos principais
df["data_reclamacao"] = pd.to_datetime(df["data_reclamacao"], errors="coerce")
df["ano"] = pd.to_numeric(df["ano"], errors="coerce").astype("Int64")
df["mes"] = pd.to_numeric(df["mes"], errors="coerce").astype("Int64")
df["casos"] = pd.to_numeric(df["casos"], errors="coerce").astype("Int64")
df["descricao_n_palavras"] = pd.to_numeric(df["descricao_n_palavras"], errors="coerce").astype("Int64")

df.shape

In [ ]:
display(df.head(3))
print("Período:", df["data_reclamacao"].min().date(), "→", df["data_reclamacao"].max().date())
print("Registros:", len(df), "| Casos (soma):", int(df["casos"].sum()))

## Visão geral (descritiva)

- Distribuição de `status`
- Top UFs por volume
- Top categorias (nível final) por volume


In [ ]:
status_dist = (
    df.groupby("status", dropna=False)["casos"].sum().sort_values(ascending=False).reset_index()
)
display(status_dist)

fig = px.bar(
    status_dist,
    x="status",
    y="casos",
    title="Distribuição de reclamações por Status (somando CASOS)",
)
fig.show()

In [ ]:
top_ufs = df.groupby(["uf", "estado"], dropna=False)["casos"].sum().sort_values(ascending=False).head(15).reset_index()
display(top_ufs)

fig = px.bar(
    top_ufs,
    x="casos",
    y="uf",
    orientation="h",
    title="Top 15 UFs por volume de reclamações",
)
fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.show()

In [ ]:
top_cat = (
    df.groupby("categoria_final", dropna=False)["casos"].sum().sort_values(ascending=False).head(20).reset_index()
)

display(top_cat)
fig = px.bar(
    top_cat,
    x="casos",
    y="categoria_final",
    orientation="h",
    title="Top 20 categorias (nível final) por volume de reclamações",
)
fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.show()

## Série temporal com tendência (média móvel)

Requisito importante para o dashboard: evolução do número de reclamações no tempo + linha de média móvel.


In [ ]:
ts_m = (
    df.groupby(pd.Grouper(key="data_reclamacao", freq="MS"))["casos"].sum().rename("casos").reset_index()
)
ts_m["mm_3m"] = ts_m["casos"].rolling(window=3, min_periods=1).mean()

display(ts_m.head(12))

ts_long = ts_m.melt(
    id_vars="data_reclamacao",
    value_vars=["casos", "mm_3m"],
    var_name="serie",
    value_name="qtd",
)

fig = px.line(
    ts_long,
    x="data_reclamacao",
    y="qtd",
    color="serie",
    markers=True,
    title="Reclamações por mês (CASOS) + Média móvel 3 meses",
)
fig.show()

## Sazonalidade

Duas visões úteis:

- Heatmap mês × ano (volume)
- Volume por dia da semana


In [ ]:
heat = (
    df.pivot_table(index="mes", columns="ano", values="casos", aggfunc="sum", fill_value=0)
    .sort_index()
)

# Seaborn/Matplotlib podem falhar com dtypes pandas 'Int64' (nullable) e/ou objetos.
# Força conversão para numérico padrão antes de plotar.
heat = heat.apply(pd.to_numeric, errors="coerce").fillna(0).astype(float)

plt.figure(figsize=(12, 5))
sns.heatmap(heat, cmap="Blues", linewidths=0.2)
plt.title("Sazonalidade: volume de reclamações (CASOS) por mês e ano")
plt.xlabel("Ano")
plt.ylabel("Mês")
plt.show()

In [ ]:
order_dow = ["Segunda", "Terça", "Quarta", "Quinta", "Sexta", "Sábado", "Domingo"]
dow = df.groupby("dia_da_semana_nome")["casos"].sum().reindex(order_dow).reset_index()

fig = px.bar(dow, x="dia_da_semana_nome", y="casos", title="Reclamações por dia da semana (CASOS)")
fig.show()

## Cruzamentos (STATUS × CATEGORIA / LOCAL)

Requisito do Acompanhamento 2: cruzar `status` com `categoria` ou `local`.

Abaixo, usamos `categoria_final` e `uf`.


In [ ]:
# Seleciona as 12 categorias mais frequentes (por CASOS)
top_cat_names = (
    df.groupby("categoria_final")["casos"].sum().sort_values(ascending=False).head(12).index
)

cat_status = (
    df[df["categoria_final"].isin(top_cat_names)]
    .groupby(["categoria_final", "status"], dropna=False)["casos"].sum()
    .reset_index()
)

fig = px.bar(
    cat_status,
    x="categoria_final",
    y="casos",
    color="status",
    title="STATUS × Categoria (Top 12) — barras empilhadas (CASOS)",
)
fig.update_layout(xaxis={"tickangle": -35})
fig.show()

In [ ]:
uf_status = (
    df.groupby(["uf", "status"], dropna=False)["casos"].sum().reset_index()
    .sort_values(["uf", "casos"], ascending=[True, False])
)

top_uf_names = df.groupby("uf")["casos"].sum().sort_values(ascending=False).head(10).index
uf_status_top = uf_status[uf_status["uf"].isin(top_uf_names)]

fig = px.bar(
    uf_status_top,
    x="uf",
    y="casos",
    color="status",
    title="STATUS × UF (Top 10 UFs) — barras empilhadas (CASOS)",
)
fig.show()

## Pareto por UF (Distribuição espacial)

Sugestão do enunciado: usar Pareto para evidenciar os estados mais críticos.


In [ ]:
uf_tot = df.groupby("uf")["casos"].sum().sort_values(ascending=False)
pareto = uf_tot.reset_index().rename(columns={"casos": "casos_uf"})
pareto["perc"] = pareto["casos_uf"] / pareto["casos_uf"].sum()
pareto["perc_acum"] = pareto["perc"].cumsum()

display(pareto.head(10))

fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.bar(pareto["uf"], pareto["casos_uf"], color="#4c78a8")
ax1.set_ylabel("CASOS por UF")
ax1.set_xlabel("UF")

ax2 = ax1.twinx()
ax2.plot(pareto["uf"], pareto["perc_acum"] * 100, color="#f58518", marker="o")
ax2.set_ylabel("% acumulado")
ax2.axhline(80, color="red", linestyle="--", linewidth=1)

plt.title("Pareto: reclamações por UF (CASOS)")
plt.show()

## Mapa do Brasil (cloroplético) com seletor de ano

O mapa abaixo já gera o efeito de **slider por ano** usando `animation_frame` do Plotly.

Obs.: o GeoJSON é baixado e cacheado em `data/ref/brazil-states.geojson` na primeira execução.


In [ ]:
GEO_DIR = PROJECT_ROOT / "data" / "ref"
GEO_DIR.mkdir(parents=True, exist_ok=True)
GEO_PATH = GEO_DIR / "brazil-states.geojson"
GEO_URL = "https://raw.githubusercontent.com/codeforamerica/click_that_hood/master/public/data/brazil-states.geojson"

if not GEO_PATH.exists():
    urllib.request.urlretrieve(GEO_URL, GEO_PATH)

with GEO_PATH.open("r", encoding="utf-8") as f:
    br_geojson = json.load(f)

map_df = df.groupby(["ano", "uf"], dropna=False)["casos"].sum().reset_index()

fig = px.choropleth(
    map_df,
    geojson=br_geojson,
    locations="uf",
    featureidkey="properties.sigla",
    color="casos",
    animation_frame="ano",
    color_continuous_scale="Reds",
    labels={"casos": "Reclamações (CASOS)", "uf": "UF"},
    title="Reclamações por UF (CASOS) — Slider por ano",
)
fig.update_geos(fitbounds="locations", visible=False)
fig.show()

## Análise estatística de textos (tamanho do texto × STATUS)

Requisito do enunciado: boxplot ou histograma mostrando a distribuição do tamanho dos textos e cruzar com `status`.


In [ ]:
plt.figure(figsize=(12, 5))
sns.boxplot(data=df, x="status", y="descricao_n_palavras")
plt.title("Distribuição do tamanho do texto (palavras) por Status")
plt.xlabel("Status")
plt.ylabel("Nº de palavras")
plt.xticks(rotation=20)
plt.show()

summary_text = df.groupby("status")["descricao_n_palavras"].describe(percentiles=[0.25, 0.5, 0.75]).round(2)
display(summary_text)

## Mineração de texto (NLP básica): WordCloud (com remoção de stopwords)

Abaixo, montamos uma nuvem de palavras a partir das descrições.

Dica: para o dashboard, é interessante permitir filtro por `uf`, `status` e `faixa_tamanho_texto`.


In [ ]:
DEFAULT_STOPWORDS_PT = {
    "a", "à", "agora", "ainda", "além", "algo", "algum", "alguma", "algumas", "alguns", "am", "ao", "aos", "após",
    "as", "até", "com", "como", "da", "das", "de", "dei", "delas", "dele", "deles", "demais", "depois", "desde",
    "do", "dos", "e", "ela", "elas", "ele", "eles", "em", "entre", "era", "eram", "essa", "essas", "esse", "esses",
    "esta", "está", "estão", "estas", "este", "estes", "eu", "faz", "foi", "for", "fora", "foram", "há", "isso",
    "isto", "já", "la", "lá", "lhe", "lhes", "mais", "mas", "me", "mesmo", "meu", "minha", "muito", "na", "não",
    "nas", "nem", "no", "nos", "nós", "o", "os", "ou", "para", "pela", "pelas", "pelo", "pelos", "por", "porque",
    "pra", "quando", "que", "quem", "se", "sem", "ser", "só", "sua", "suas", "também", "tem", "tendo", "tenho",
    "ter", "teve", "tinha", "tive", "toda", "todas", "todo", "todos", "um", "uma", "umas", "uns", "vai", "vou",
    "vocês", "você", "vc", "vcs", "www", "comprei", "produto", "loja", "ibyte",
}


def normalize_text(text: str) -> str:
    text = text.lower()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


all_text = " ".join(df["descricao"].dropna().astype(str).tolist())
tokens = [
    t
    for t in normalize_text(all_text).split()
    if (t not in DEFAULT_STOPWORDS_PT) and (len(t) >= 3)
]

freq = Counter(tokens)
top_words = pd.DataFrame(freq.most_common(20), columns=["palavra", "freq"]) 
display(top_words)

wc = WordCloud(
    width=1200,
    height=500,
    background_color="white",
    colormap="viridis",
)
wc.generate_from_frequencies(freq)

plt.figure(figsize=(14, 6))
plt.imshow(wc, interpolation="bilinear")
plt.axis("off")
plt.title("WordCloud — termos mais frequentes nas descrições")
plt.show()

## Rascunho de métricas para o Dashboard

KPIs e métricas recomendadas (podem virar cards/tabelas no BI):

- Total de reclamações (**CASOS**) no período
- % Resolvido (CASOS resolvidos / CASOS total)
- Top UF e Top Categoria
- Tendência: média móvel (3M ou 6M)

Filtros globais obrigatórios no dashboard (do enunciado):

- **Estado/UF**
- **Status**
- **Faixa de tamanho do texto** (`faixa_tamanho_texto`)


In [ ]:
kpis = {
    "registros": int(len(df)),
    "casos_total": int(df["casos"].sum()),
    "pct_resolvido": float(
        (df.loc[df["status"] == "Resolvido", "casos"].sum() / df["casos"].sum())
        if df["casos"].sum() else 0
    ),
    "top_uf": str(df.groupby("uf")["casos"].sum().sort_values(ascending=False).index[0]),
    "top_categoria": str(df.groupby("categoria_final")["casos"].sum().sort_values(ascending=False).index[0]),
}

pd.DataFrame([kpis])

## Exportando agregações (opcional, mas ajuda no dashboard)

Gera tabelas já prontas para consumo no Streamlit/Dash:

- Série temporal mensal com média móvel
- CASOS por UF e ano (para o mapa)
- CASOS por status
- CASOS por categoria e status


In [ ]:
OUT_DIR = PROJECT_ROOT / "data" / "prc"
OUT_DIR.mkdir(parents=True, exist_ok=True)

agg_status = df.groupby("status", dropna=False)["casos"].sum().reset_index()
agg_uf_ano = df.groupby(["ano", "uf"], dropna=False)["casos"].sum().reset_index()
agg_cat_status = df.groupby(["categoria_final", "status"], dropna=False)["casos"].sum().reset_index()

ts_m.to_csv(OUT_DIR / "agg_mensal_casos.csv", index=False, encoding="utf-8")
agg_status.to_csv(OUT_DIR / "agg_status_casos.csv", index=False, encoding="utf-8")
agg_uf_ano.to_csv(OUT_DIR / "agg_uf_ano_casos.csv", index=False, encoding="utf-8")
agg_cat_status.to_csv(OUT_DIR / "agg_categoria_status_casos.csv", index=False, encoding="utf-8")

(OUT_DIR / "agg_mensal_casos.csv", OUT_DIR / "agg_uf_ano_casos.csv")